In [0]:
catalog = "dbr_dev_ua5816bd"
source_schema = "mialkovska_viktor594"
bronze_schema = "mialkovska_viktor594_bronze"
volume = "raw_files"

source_path = f"/Volumes/{catalog}/{source_schema}/{volume}"
dataset_path = f"{source_path}/tristar"

staging_path = f"{dataset_path}/staging"
schema_path = f"{dataset_path}/schema"
checkpoint_path = f"{dataset_path}/checkpoint"

bronze_table = f"{catalog}.{bronze_schema}.tristar"

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .option("pathGlobFilter", "*.json")
        .option("multiLine", "true")
        .load(staging_path)
        .withColumn("source_filename", col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
)

In [0]:
from delta.tables import DeltaTable

query = (
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(once=True)
        .table(bronze_table)
)
query.awaitTermination()

In [0]:
display(spark.table(bronze_table))

In [0]:
spark.table(bronze_table).printSchema()

In [0]:
source_files = len(dbutils.fs.ls(staging_path))
source_files

In [0]:
bronze_count = spark.table(bronze_table).count()
bronze_count

In [0]:
from pyspark.sql.functions import countDistinct

spark.table(bronze_table) \
    .select(countDistinct("source_filename").alias("files_ingested")) \
    .show()

In [0]:
spark.table(bronze_table) \
    .select(
        "vehicleId",
        "speed",
        "lat",
        "_rescued_data",
        "source_filename"
    ) \
    .where("_rescued_data IS NOT NULL") \
    .show(truncate=False)

In [0]:
spark.sql(f"DESCRIBE HISTORY {bronze_table}").display()

In [0]:
query.recentProgress